# Chunking, Embedding, and Indexing

This notebook converts the raw SEC corpus into four chunking variants (fixed, section, semantic, hierarchical), embeds every chunk, and persists a dense FAISS + sparse BM25 index for each variant.

The artifacts are written in a uniform layout so the evaluation notebook can load any strategy and swap retrievers without special-casing.

In [1]:
import json
import time
import pickle
import re
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import tiktoken
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from tqdm.auto import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PROCESSED = ROOT / 'data' / 'processed'
ARTIFACTS = ROOT / 'artifacts'
ARTIFACTS.mkdir(parents=True, exist_ok=True)

STRATEGIES = ['fixed', 'section', 'semantic', 'hierarchical']
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
TARGET_FILING_YEAR = 2025


def load_corpus(path: Path | None = None) -> pd.DataFrame:
    path = path or (DATA_PROCESSED / 'sec_corpus.jsonl')
    rows = []
    with path.open('r', encoding='utf-8') as handle:
        for line in handle:
            rows.append(json.loads(line))
    corpus = pd.DataFrame(rows)
    if 'filing_date' not in corpus.columns:
        return corpus
    filing_dates = pd.to_datetime(corpus['filing_date'], errors='coerce')
    return corpus.loc[filing_dates.dt.year == TARGET_FILING_YEAR].reset_index(drop=True)


def clean_text(text: str) -> str:
    text = re.sub(r'\s+', ' ', text or '')
    return text.strip()


ENCODER = tiktoken.get_encoding('cl100k_base')


def token_len(text: str) -> int:
    """Return the number of tokens in text using cl100k_base encoding."""
    return len(ENCODER.encode(text))


def token_chunk(text: str, chunk_tokens: int = 128, overlap_tokens: int = 32) -> list[str]:
    """
    Split text into token-aware chunks of `chunk_tokens` with `overlap_tokens` overlap.
    This is the leaf-level splitter used by the recursive chunker.

    Args:
        text:           Input text string.
        chunk_tokens:   Target number of tokens per chunk. Default 128.
        overlap_tokens: Number of tokens to re-include from the previous chunk. Default 32.

    Returns:
        List of decoded text chunks.
    """
    tokens = ENCODER.encode(clean_text(text))
    if not tokens:
        return []
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(len(tokens), start + chunk_tokens)
        chunks.append(ENCODER.decode(tokens[start:end]))
        if end == len(tokens):
            break
        start = max(0, end - overlap_tokens)
    return chunks


def fixed_chunk(text: str, chunk_tokens: int = 128, overlap_tokens: int = 32) -> list[str]:
    """
    Token-aware fixed-size chunker. 

    Args:
        text:           Input text.
        chunk_tokens:   Tokens per chunk. Default 128 (~800 chars).
        overlap_tokens: Overlap tokens. Default 32 (~200 chars).
    """
    return token_chunk(text, chunk_tokens=chunk_tokens, overlap_tokens=overlap_tokens)


SECTION_KEYWORDS = [
    r'part\s+[ivx]+\b(?:\s*[:.-]\s*[a-z][a-z\s]{0,120})?',
    r'item\s+\d{1,2}[a-z]?\b(?:\s*[:.-]\s*[a-z][a-z\s]{0,160})?',
    r'management(?:[\u2019\']s)?\s+discussion\s+and\s+analysis',
    r'financial\s+statements(?:\s+and\s+supplementary\s+data)?',
    r'controls\s+and\s+procedures',
    r'quantitative\s+and\s+qualitative\s+disclosures\s+about\s+market\s+risk',
]
SECTION_TOKEN_PATTERN = '(?:' + '|'.join(SECTION_KEYWORDS) + ')'
SECTION_PREFIX_PATTERN = re.compile(rf'(?i)(<!\n)\s*({SECTION_TOKEN_PATTERN})')
SECTION_PATTERN = re.compile(rf'(?im)^\s*({SECTION_TOKEN_PATTERN})')


def normalize_for_sectioning(text: str) -> str:
    text = (text or '').replace('\r\n', '\n').replace('\r', '\n')
    text = re.sub(r'[ \t\f\v]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def push_section_prefixes(text: str) -> str:
    # SEC extracts often flatten headings inline; add synthetic line breaks before headings.
    return SECTION_PREFIX_PATTERN.sub(lambda match: f"\n{match.group(1)}", text)


def section_chunk(text: str, fallback_to_fixed: bool = True, min_section_chars: int = 80) -> list[str]:
    normalized = normalize_for_sectioning(text)
    if not normalized:
        return []

    prepared = push_section_prefixes(normalized)
    matches = list(SECTION_PATTERN.finditer(prepared))
    if not matches:
        return fixed_chunk(normalized) if fallback_to_fixed else []

    boundaries = sorted({0, *(m.start() for m in matches), len(prepared)})
    sections = []
    for start, end in zip(boundaries, boundaries[1:]):
        section = clean_text(prepared[start:end])
        if section and len(section) >= min_section_chars:
            sections.append(section)

    if sections:
        return sections
    return fixed_chunk(normalized) if fallback_to_fixed else []


# -- Recursive parent-child section chunker ----------------------------------

PARENT_MAX_TOKENS = 512    # a section <= this is kept as a parent and split into children
CHILD_TARGET_TOKENS = 128  # target size of each child chunk
CHILD_OVERLAP_TOKENS = 32  # overlap between children


def recursive_section_chunk(
    text: str,
    parent_max_tokens: int = PARENT_MAX_TOKENS,
    child_target_tokens: int = CHILD_TARGET_TOKENS,
    child_overlap_tokens: int = CHILD_OVERLAP_TOKENS,
    depth: int = 0,
    max_depth: int = 4,
) -> list[dict]:
    """
    Recursively splits a text block into parent-child chunk pairs.

    Logic:
      1. If the text is already small (<=child_target_tokens), return it as a leaf.
      2. If the text fits within parent_max_tokens, split it into token_chunk children
         and attach the full text as parent_text on each child.
      3. If the text exceeds parent_max_tokens, first try to split it by SEC section
         headings. Recurse into each sub-section.
      4. If no section headings are found, force-split into parent_max_token sized
         pieces using token_chunk, then recurse into each piece.

    Args:
        text:               The text block to split.
        parent_max_tokens:  Max tokens a node can be before it must be split further.
                            Default 512.
        child_target_tokens:Target tokens for leaf chunks. Default 128.
        child_overlap_tokens:Overlap tokens for leaf chunks. Default 32.
        depth:              Current recursion depth (do not set manually).
        max_depth:          Maximum recursion depth to prevent infinite loops. Default 4.

    Returns:
        List of dicts, each with keys:
            'text'        - the child chunk text (this is what gets embedded)
            'parent_text' - the parent section text (this is what gets sent to the LLM)
            'level'       - recursion depth at which this chunk was created
    """
    text = clean_text(text)
    if not text:
        return []

    tlen = token_len(text)

    # Base case: text is already small enough to be a leaf
    if tlen <= child_target_tokens or depth >= max_depth:
        return [{'text': text, 'parent_text': text, 'level': depth}]

    # Node fits as a parent: split into children and attach parent_text
    if tlen <= parent_max_tokens:
        children = token_chunk(text, chunk_tokens=child_target_tokens, overlap_tokens=child_overlap_tokens)
        return [
            {'text': child, 'parent_text': text, 'level': depth + 1}
            for child in children
        ]

    # Node is too large: try section-heading split first
    sub_sections = section_chunk(text, fallback_to_fixed=False, min_section_chars=80)

    if sub_sections and len(sub_sections) > 1:
        # Recurse into each sub-section
        results = []
        for sub in sub_sections:
            results.extend(
                recursive_section_chunk(
                    sub,
                    parent_max_tokens=parent_max_tokens,
                    child_target_tokens=child_target_tokens,
                    child_overlap_tokens=child_overlap_tokens,
                    depth=depth + 1,
                    max_depth=max_depth,
                )
            )
        return results

    # No useful section boundaries found: force-split into parent-sized pieces, then recurse
    forced_parents = token_chunk(text, chunk_tokens=parent_max_tokens, overlap_tokens=0)
    results = []
    for piece in forced_parents:
        results.extend(
            recursive_section_chunk(
                piece,
                parent_max_tokens=parent_max_tokens,
                child_target_tokens=child_target_tokens,
                child_overlap_tokens=child_overlap_tokens,
                depth=depth + 1,
                max_depth=max_depth,
            )
        )
    return results


def sentence_split(text: str) -> list[str]:
    sentences = re.split(r'(?<=[.!?])\s+', clean_text(text))
    return [sentence for sentence in sentences if sentence]


def semantic_chunk(
    text: str,
    model: SentenceTransformer,
    similarity_threshold: float = 0.70,
    max_tokens: int = 256,
    window_size: int = 2,
) -> list[str]:
    """
    Splits text into semantically coherent chunks using a rolling embedding window.

    Instead of comparing only adjacent sentences (i-1 vs i), this computes the
    mean embedding of the last `window_size` sentences and compares it to the
    next sentence. This catches gradual topic shifts that span multiple sentences.

    Args:
        text:                 Input text.
        model:                SentenceTransformer model for encoding.
        similarity_threshold: Cosine similarity below which a new chunk starts.
                              Default 0.70 (lower = larger chunks).
                              Tune between 0.35-0.70.
        max_tokens:           Hard token ceiling per chunk. Default 256.
                              Tune between 128-512.
        window_size:          Number of recent sentence embeddings to average
                              for the rolling context window. Default 2.
                              Tune between 1-4.

    Returns:
        List of chunk text strings.
    """
    sentences = sentence_split(text)
    if not sentences:
        return []
    if len(sentences) == 1:
        return token_chunk(sentences[0], chunk_tokens=max_tokens, overlap_tokens=0)

    embeddings = model.encode(sentences, normalize_embeddings=True, show_progress_bar=False)

    chunks = []
    current_sentences = [sentences[0]]
    current_indices = [0]
    current_tokens = token_len(sentences[0])

    for idx in range(1, len(sentences)):
        next_sentence = sentences[idx]
        next_tokens = token_len(next_sentence)

        window_indices = current_indices[-max(window_size, 1):]
        window_embedding = np.mean(embeddings[window_indices], axis=0)
        similarity = float(np.dot(window_embedding, embeddings[idx]))

        should_split = similarity < similarity_threshold or (current_tokens + next_tokens > max_tokens)
        if should_split:
            chunk_text = ' '.join(current_sentences).strip()
            if token_len(chunk_text) > max_tokens:
                chunks.extend(token_chunk(chunk_text, chunk_tokens=max_tokens, overlap_tokens=0))
            else:
                chunks.append(chunk_text)
            current_sentences = [next_sentence]
            current_indices = [idx]
            current_tokens = next_tokens
        else:
            current_sentences.append(next_sentence)
            current_indices.append(idx)
            current_tokens += next_tokens

    if current_sentences:
        chunk_text = ' '.join(current_sentences).strip()
        if token_len(chunk_text) > max_tokens:
            chunks.extend(token_chunk(chunk_text, chunk_tokens=max_tokens, overlap_tokens=0))
        else:
            chunks.append(chunk_text)

    return [chunk for chunk in chunks if chunk]


def hierarchical_chunk(
    text: str,
    model: SentenceTransformer | None = None,
    section_max_chars: int = 8000,
) -> list[str]:
    sections = section_chunk(text, fallback_to_fixed=False)
    if not sections:
        if model is not None:
            return semantic_chunk(text, model=model)
        return fixed_chunk(text)

    chunks = []
    for section in sections:
        if len(section) <= section_max_chars:
            chunks.append(section)
        elif model is not None:
            chunks.extend(semantic_chunk(section, model=model))
        else:
            chunks.extend(fixed_chunk(section))
    return chunks


def build_chunk_table(corpus: pd.DataFrame, strategy: str, model: SentenceTransformer | None = None, show_progress: bool = True) -> pd.DataFrame:
    chunk_rows = []
    iterator = corpus.itertuples(index=False)
    if show_progress:
        iterator = tqdm(
            iterator,
            total=len(corpus),
            desc=f'chunk[{strategy}]',
            unit='filing',
            leave=False,
            bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]',
        )
    for row in iterator:
        text = str(row.text)
        chunk_entries = []

        if strategy == 'fixed':
            chunk_entries = [{'text': c, 'parent_text': text, 'level': 0} for c in fixed_chunk(text)]
        elif strategy == 'section':
            chunk_entries = recursive_section_chunk(text)
        elif strategy == 'semantic':
            if model is None:
                raise ValueError('`model` is required for semantic chunking.')
            chunk_entries = [{'text': c, 'parent_text': text, 'level': 0} for c in semantic_chunk(text, model=model)]
        elif strategy == 'hierarchical':
            chunk_entries = [{'text': c, 'parent_text': text, 'level': 0} for c in hierarchical_chunk(text, model=model)]
        else:
            raise ValueError(f'Unknown strategy: {strategy}')

        for idx, entry in enumerate(chunk_entries):
            child_text = entry.get('text', '')
            parent_text = entry.get('parent_text', child_text)
            level = int(entry.get('level', 0))
            if not child_text:
                continue
            chunk_rows.append({
                'chunk_id': f'{row.ticker}-{row.form}-{row.filing_date}-{strategy}-{idx}',
                'strategy': strategy,
                'company': str(row.company),
                'ticker': str(row.ticker),
                'form': str(row.form),
                'filing_date': str(row.filing_date),
                'url': str(row.url),
                'chunk_index': idx,
                'text': child_text,
                'child_text': child_text,
                'parent_text': parent_text,
                'level': level,
            })
    return pd.DataFrame(chunk_rows)


def embed_texts(texts: list[str], model: SentenceTransformer, batch_size: int = 64, show_progress: bool = True) -> np.ndarray:
    embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=show_progress, batch_size=batch_size)
    return np.asarray(embeddings, dtype='float32')


def build_faiss_index(embeddings: np.ndarray) -> faiss.Index:
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return index


BM25_STOPWORDS = {
    'a', 'an', 'and', 'are', 'as', 'at', 'be', 'by', 'for', 'from', 'has', 'he',
    'in', 'is', 'it', 'its', 'of', 'on', 'that', 'the', 'to', 'was', 'were', 'will', 'with',
}


def tokenize_for_bm25(text: str) -> list[str]:
    tokens = re.findall(r'[a-z0-9]+', clean_text(text).lower())
    filtered = [token for token in tokens if token not in BM25_STOPWORDS]
    return filtered or ['_']


def build_bm25_index(texts: list[str]) -> BM25Okapi:
    return BM25Okapi([tokenize_for_bm25(t) for t in texts])


def persist_strategy(strategy: str, chunks: pd.DataFrame, embeddings: np.ndarray, index: faiss.Index, bm25: BM25Okapi) -> None:
    # Stream JSONL row-by-row to avoid building one large in-memory JSON string.
    out_path = DATA_PROCESSED / f'chunks_{strategy}.jsonl'
    with out_path.open('w', encoding='utf-8') as handle:
        for record in chunks.to_dict(orient='records'):
            handle.write(json.dumps(record, ensure_ascii=False) + '\n')

    faiss.write_index(index, str(ARTIFACTS / f'{strategy}_faiss.index'))
    np.save(ARTIFACTS / f'{strategy}_embeddings.npy', embeddings)
    with (ARTIFACTS / f'{strategy}_bm25.pkl').open('wb') as handle:
        pickle.dump(bm25, handle)
    meta_cols = ['chunk_id', 'strategy', 'company', 'ticker', 'form', 'filing_date', 'url', 'level']
    chunks[meta_cols].to_json(ARTIFACTS / f'{strategy}_chunk_metadata.json', orient='records', indent=2)

In [2]:
_build_chunk_table = build_chunk_table

MIN_CHARS_BY_STRATEGY = {
    'fixed': 100,
    'section': 50,
    'semantic': 50,
    'hierarchical': 100,
}


def filter_chunks(table: pd.DataFrame, strategy: str, min_chars: int | None = None) -> pd.DataFrame:
    min_chars = MIN_CHARS_BY_STRATEGY.get(strategy, 50) if min_chars is None else min_chars
    if table.empty:
        return table

    lengths = table['text'].str.len()
    filtered = table.loc[lengths >= min_chars].reset_index(drop=True)
    dropped = len(table) - len(filtered)
    print(f'[{strategy}] dropped {dropped} chunks below {min_chars} chars')
    return filtered


def build_chunk_table(corpus: pd.DataFrame, strategy: str, model: SentenceTransformer | None = None, show_progress: bool = True) -> pd.DataFrame:
    table = _build_chunk_table(corpus, strategy, model=model, show_progress=show_progress)
    return filter_chunks(table, strategy)


In [3]:
corpus = load_corpus()
print(f'Loaded {len(corpus)} filings for year {TARGET_FILING_YEAR}.')
total_chars = int(corpus['text'].str.len().sum()) if len(corpus) else 0
print(f'Total characters across filings: {total_chars:,}')
corpus[['company', 'ticker', 'form', 'filing_date']]


Loaded 24 filings for year 2025.
Total characters across filings: 1,902,941


,company,ticker,form,filing_date
0,Visa Inc.,V,10-K,2025-11-06
1,Visa Inc.,V,10-Q,2025-07-30
2,Visa Inc.,V,10-Q,2025-04-30
3,Visa Inc.,V,10-Q,2025-01-31
4,Visa Inc.,V,8-K,2025-12-30
5,Visa Inc.,V,8-K,2025-12-23
6,Visa Inc.,V,8-K,2025-11-10
7,Visa Inc.,V,8-K,2025-10-28
8,Visa Inc.,V,8-K,2025-09-29
9,Visa Inc.,V,8-K,2025-09-26


In [4]:
t0 = time.perf_counter()
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f'Loaded {EMBEDDING_MODEL_NAME} in {time.perf_counter() - t0:.1f}s')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded sentence-transformers/all-MiniLM-L6-v2 in 3.0s


In [5]:
# Build chunk tables for every strategy. Semantic/hierarchical call model.encode
# per filing on CPU, so expect this step to dominate the runtime. Timing is
# printed per strategy so a stall is obvious.
chunk_tables: dict[str, pd.DataFrame] = {}
for strategy in STRATEGIES:
    t0 = time.perf_counter()
    table = build_chunk_table(
        corpus,
        strategy,
        model=embedding_model if strategy in {'semantic', 'hierarchical'} else None,
        show_progress=True,
    )
    chunk_tables[strategy] = table
    print(f'[{strategy}] {len(table):>5} chunks in {time.perf_counter() - t0:.1f}s')


chunk[fixed]:   0%|          | 0/24 [00:00<?]

[fixed] dropped 0 chunks below 100 chars
[fixed]  4625 chunks in 0.5s


chunk[section]:   0%|          | 0/24 [00:00<?]

[section] dropped 85 chunks below 50 chars
[section]  4341 chunks in 2.0s


chunk[semantic]:   0%|          | 0/24 [00:00<?]

[semantic] dropped 1230 chunks below 50 chars
[semantic]  7609 chunks in 109.9s


chunk[hierarchical]:   0%|          | 0/24 [00:00<?]

[hierarchical] dropped 2703 chunks below 100 chars
[hierarchical]  6136 chunks in 108.7s


In [6]:
strategy = 'fixed'
table = chunk_tables[strategy]
t0 = time.perf_counter()
embeddings = embed_texts(table['text'].tolist(), embedding_model, show_progress=True)
print(f'[{strategy}] embedded {len(table)} chunks in {time.perf_counter() - t0:.1f}s')

t0 = time.perf_counter()
index = build_faiss_index(embeddings)
bm25 = build_bm25_index(table['text'].tolist())
persist_strategy(strategy, table, embeddings, index, bm25)
print(f'[{strategy}] indexed + persisted in {time.perf_counter() - t0:.1f}s')


Batches:   0%|          | 0/73 [00:00<?, ?it/s]

[fixed] embedded 4625 chunks in 128.0s
[fixed] indexed + persisted in 13.4s


In [7]:
strategy = 'section'
table = chunk_tables[strategy]
t0 = time.perf_counter()
embeddings = embed_texts(table['text'].tolist(), embedding_model, show_progress=True)
print(f'[{strategy}] embedded {len(table)} chunks in {time.perf_counter() - t0:.1f}s')

t0 = time.perf_counter()
index = build_faiss_index(embeddings)
bm25 = build_bm25_index(table['text'].tolist())
persist_strategy(strategy, table, embeddings, index, bm25)
print(f'[{strategy}] indexed + persisted in {time.perf_counter() - t0:.1f}s')


Batches:   0%|          | 0/68 [00:00<?, ?it/s]

[section] embedded 4341 chunks in 122.4s
[section] indexed + persisted in 1.0s


In [8]:
strategy = 'semantic'
table = chunk_tables[strategy]
t0 = time.perf_counter()
embeddings = embed_texts(table['text'].tolist(), embedding_model, show_progress=True)
print(f'[{strategy}] embedded {len(table)} chunks in {time.perf_counter() - t0:.1f}s')

t0 = time.perf_counter()
index = build_faiss_index(embeddings)
bm25 = build_bm25_index(table['text'].tolist())
persist_strategy(strategy, table, embeddings, index, bm25)
print(f'[{strategy}] indexed + persisted in {time.perf_counter() - t0:.1f}s')


Batches:   0%|          | 0/119 [00:00<?, ?it/s]

[semantic] embedded 7609 chunks in 117.7s
[semantic] indexed + persisted in 24.3s


In [9]:
strategy = 'hierarchical'
table = chunk_tables[strategy]
t0 = time.perf_counter()
embeddings = embed_texts(table['text'].tolist(), embedding_model, show_progress=True)
print(f'[{strategy}] embedded {len(table)} chunks in {time.perf_counter() - t0:.1f}s')

t0 = time.perf_counter()
index = build_faiss_index(embeddings)
bm25 = build_bm25_index(table['text'].tolist())
persist_strategy(strategy, table, embeddings, index, bm25)
print(f'[{strategy}] indexed + persisted in {time.perf_counter() - t0:.1f}s')


Batches:   0%|          | 0/96 [00:00<?, ?it/s]

[hierarchical] embedded 6136 chunks in 111.3s
[hierarchical] indexed + persisted in 20.4s


In [10]:
def chunk_statistics(strategy: str, chunks: pd.DataFrame) -> dict:
    lengths = chunks['text'].str.len()
    return {
        'strategy': strategy,
        'num_chunks': int(len(chunks)),
        'num_filings_covered': int(chunks.groupby(['ticker', 'form', 'filing_date']).ngroups),
        'mean_chars': round(float(lengths.mean()) if len(lengths) else 0.0, 1),
        'median_chars': round(float(lengths.median()) if len(lengths) else 0.0, 1),
        'p95_chars': round(float(lengths.quantile(0.95)) if len(lengths) else 0.0, 1),
        'min_chars': int(lengths.min()) if len(lengths) else 0,
        'max_chars': int(lengths.max()) if len(lengths) else 0,
        'total_chars': int(lengths.sum()),
    }

stats_df = pd.DataFrame([chunk_statistics(s, chunk_tables[s]) for s in STRATEGIES])
stats_df.to_csv(DATA_PROCESSED / 'chunk_stats.csv', index=False)
stats_df

,strategy,num_chunks,num_filings_covered,mean_chars,median_chars,p95_chars,min_chars,max_chars,total_chars
0,fixed,4625,24,548.3,560.0,777.8,169,920,2535845
1,section,4341,24,547.8,562.0,779.0,144,895,2378080
2,semantic,7609,24,245.0,178.0,714.6,50,1592,1864299
3,hierarchical,6136,24,286.1,206.0,759.0,100,1592,1755244


Every strategy now has its own FAISS index, BM25 index, and metadata file under `artifacts/`. The evaluation notebook reads those artifacts to compare retrieval quality across strategies on identical queries.